# HHMARL 2D in JAX: massively parallel reproduction on a single Kaggle GPU

This notebook is a self-contained JAX re-implementation of **IDSIA/hhmarl_2D** (Selmonaj et al.,
*Hierarchical Multi-Agent Reinforcement Learning for Air Combat Maneuvering*, arXiv 2309.11247, ICMLA 2023).
It replaces the original Python simulator and the Ray RLlib PPO setup with:

* a fully `jit` + `vmap` compiled environment (low-level fight/escape env with the L1–L5 curriculum,
  scripted and fictitious-self-play opponents, heterogeneous AC1/AC2 aircraft; and the hierarchical
  commander env that calls frozen low-level policies), so thousands of combats run in parallel on one GPU;
* the original networks (self-attention fight nets, escape nets, GRU commander, centralized critics) and
  PPO with the original RLlib hyper-parameters, including (behind `RLLIB_FAITHFUL`) the RLlib training quirks;
* training runs for the paper experiments (Fig. 6, 8, 10), FSP levels L4/L5, evaluation (Fig. 7, 8b, 10b)
  and paper-style figures with the paper's numbers overlaid.

**Defaults follow the original code, not the paper.** Paper-vs-code differences are switches in the config cell.

How to use on Kaggle: enable a GPU (T4 x1 recommended), choose `PRESET`, `BUDGET` and `EXPERIMENTS` in the
config cell, then *Run All*. Every stage writes checkpoints, CSV logs and figures to `OUT_DIR`. A 12 h session
limit is handled by `MAX_SESSION_HOURS`: the notebook checkpoints and stops cleanly; attach the output of the
previous run as an input dataset and run again to continue (see `RESUME_DIRS`).

Sections: 1 Setup · 2 Config · 3 JAX env · 4 Networks + PPO · 5 Training runs · 6 Evaluation ·
7 Figures · 8 Known deviations.

## 1. Setup
Installs JAX with CUDA 12 wheels on Kaggle (skipped locally or when `HHMARL_SKIP_PIP=1`), checks the
accelerator and creates the output folders. `OUT_DIR` defaults to `/kaggle/working` and falls back to
`./outputs` when not on Kaggle (override with the env var `HHMARL_OUT_DIR`).

In [ ]:
import os, sys, subprocess, time, json, glob, shutil, pickle, math, csv, functools
IN_KAGGLE = os.path.isdir('/kaggle/working')
if IN_KAGGLE and os.environ.get('HHMARL_SKIP_PIP', '0') != '1':
    # jax[cuda12] brings its own CUDA/cuDNN wheels; optax for Adam; pillow for GIF export.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'jax[cuda12]', 'optax',
                    'matplotlib', 'pillow'], check=False)

import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('python', sys.version.split()[0], '| jax', jax.__version__, '| optax', optax.__version__)
print('devices:', jax.devices())
BACKEND = jax.default_backend()
if BACKEND != 'gpu':
    print('WARNING: no GPU found, running on', BACKEND, '(fine for SMOKE runs, far too slow for full training)')
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip())
except Exception:
    pass

OUT_DIR = os.environ.get('HHMARL_OUT_DIR') or ('/kaggle/working' if IN_KAGGLE else os.path.abspath('./outputs'))
DIRS = {k: os.path.join(OUT_DIR, k) for k in ['checkpoints', 'policies', 'logs', 'figures', 'eval']}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
SESSION_START = time.time()
print('OUT_DIR =', OUT_DIR)

## 2. Configuration

* **`SMOKE`**: tiny CPU-sized run of every stage (a few hundred env steps each). Keep `False` for real runs.
* **`PRESET`**:
  * `faithful`: the original RLlib batch geometry. 2000 env steps per PPO update for the low level and
    1000 for the commander, minibatch 256, 30 SGD epochs, few envs. Slow on a GPU because it is sequential
    SGD-bound, but it reproduces the original sample reuse.
  * `throughput`: about 1000 parallel envs and large batches, for GPU speed. It has fewer gradient steps per env step, so
    learning curves per env step will differ from the paper.
* **`BUDGET`**: env steps per stage (`demo` about 1 GPU-hour in total, `full` at about the paper's scale; see the table).
  `BUDGET_SCALE` multiplies all stage budgets.
* **`EXPERIMENTS`**: which parts to run. Stages that are already finished in `OUT_DIR` (or in a resumed input
  dataset) are skipped automatically.
* **Paper-vs-code switches**: every default is the ORIGINAL CODE value. Set a `PAPER_*` switch to `True` to use the
  value stated in the paper instead.
* **`RLLIB_FAITHFUL`**: replicates the RLlib training quirks documented in the spec. With `True`:
  * the fight self-attention runs over ≤ 20-step chunks *including future steps* and zero-padded rows during
    SGD, while rollouts use a length-1 sequence;
  * the unmasked loss includes the padded rows;
  * the centralized critic sees zero actions at rollout/GAE time and the real actions during training;
  * the commander GRU state is reset each step and chained across agents at evaluation.

  With `False`, the clean versions are used.

In [ ]:
SMOKE = False                      # tiny end-to-end check (CPU friendly). Keep False for real runs.
if os.environ.get('HHMARL_SMOKE') is not None:
    SMOKE = os.environ['HHMARL_SMOKE'] == '1'

PRESET = 'faithful'                # 'faithful' | 'throughput'
BUDGET = 'demo'                    # 'demo' | 'full'
BUDGET_SCALE = 1.0                 # multiplies every stage budget
EXPERIMENTS = ['fig6', 'fig8', 'fsp', 'fig10', 'eval', 'figures']   # subset of these, in this order
SEED = 0
MAX_SESSION_HOURS = 11.3           # stop training cleanly (checkpoint) before Kaggle's 12 h limit
CKPT_EVERY_MIN = 20                # periodic checkpoint inside a stage (minutes)
# Previous outputs to resume from (attach a previous run's /kaggle/working as an input dataset).
RESUME_DIRS = sorted(glob.glob('/kaggle/input/*')) + sorted(glob.glob('/kaggle/input/*/*'))

# ---- RLlib training quirks -------------------------------------------------------------------
RLLIB_FAITHFUL = True

# ---- paper-vs-code switches (False = original code, the default) ----------------------------
PAPER_PPO = False          # gamma 0.95 / clip 0.2 (paper) instead of gamma 0.99 / clip 0.25 (code)
PAPER_HORIZONS = False     # L1..L5 horizons 200/250/300/350/400 (paper) instead of 150/200/300/350/400
PAPER_HIT_PROB = False     # cannon hit prob per burst 0.70/0.85 (paper) instead of 0.75/0.90
PAPER_HL_OOB = False       # commander out-of-bounds penalty -5 (paper) instead of -2
PAPER_L5_POOL = False      # L5 opponents = L1..L4 fight policies (paper) instead of {L3 fight, L4 fight, L3 escape}
ESC_DIST_REW = False       # code flag `esc_dist_rew` (per-step distance reward for escape; default off)
FRIENDLY_KILL = True       # code default: cannon/rockets can hit teammates
FRIENDLY_PUNISH = False    # code default: victim of friendly fire not punished
HIER_OPP_FIGHT_RATIO = 75  # % of commander-env opponents choosing the fight policy
HIER_ACTION_ASSESS = True  # commander situation reward (+0.1 / -0.1)
HL_OPP_START_SPEED0 = True # code quirk: args.level defaults to 1 in commander mode -> opponents start at speed 0
ESCAPE_L5_STAGE = False    # optional README step: train escape again at L5 vs L5 fight
FIG6_FC_CURRICULUM = False # FC baseline of Fig. 6 trained L1->L3 (True) or at L3 from scratch (False)

# ---- PPO hyper-parameters (train_hetero.py / train_hier.py + RLlib 2.4 defaults) -------------
PPO_LOW = dict(lr=1e-4, gamma=0.99, lam=0.95, clip=0.25, kl_coeff=0.2, kl_target=0.025,
               vf_coeff=1.0, vf_clip=10.0, ent_coeff=0.0, max_seq_len=20)
PPO_CMD = dict(lr=1e-4, gamma=0.99, lam=1.0, clip=0.25, kl_coeff=0.2, kl_target=0.05,
               vf_coeff=1.0, vf_clip=10.0, ent_coeff=0.0, max_seq_len=20)
if PAPER_PPO:
    for _p in (PPO_LOW, PPO_CMD):
        _p.update(gamma=0.95, clip=0.2)

PRESETS = {
    # n_envs * rollout = env steps per PPO update (RLlib: 2000 low level, 1000 commander)
    'faithful':   dict(low_envs=10, low_rollout=200, low_minibatch=256, low_epochs=30,
                       cmd_envs=10, cmd_rollout=100, cmd_minibatch=256, cmd_epochs=30),
    'throughput': dict(low_envs=1024, low_rollout=40, low_minibatch=4096, low_epochs=10,
                       cmd_envs=256, cmd_rollout=20, cmd_minibatch=1536, cmd_epochs=10),
}
HP = dict(PRESETS[PRESET])

# env steps per stage (commander: commander steps, as on the paper's Fig. 10 x-axis)
BUDGETS = {
    'demo': dict(fight_L1=1.5e6, fight_L2=1.5e6, fight_L3=3e6, fight_L3_nocurr=3e6, fight_L3_fc=3e6,
                 escape_L3=1.5e6, fight_L4=1.5e6, fight_L5=1.5e6, escape_L5=1e6,
                 commander_gru=0.5e6, commander_sa=0.5e6, commander_fc=0.5e6),
    # about 10k RLlib iterations per level (code default), L3/escape/commander lengths read off the paper axes
    'full': dict(fight_L1=20e6, fight_L2=20e6, fight_L3=28e6, fight_L3_nocurr=28e6, fight_L3_fc=28e6,
                 escape_L3=8.2e6, fight_L4=20e6, fight_L5=20e6, escape_L5=8e6,
                 commander_gru=6.7e6, commander_sa=6.7e6, commander_fc=6.7e6),
}
STAGE_BUDGET = {k: int(v * BUDGET_SCALE) for k, v in BUDGETS[BUDGET].items()}

EVAL_EPISODES = 1000               # paper: 1000 episodes per scenario
EVAL_SIZES = [2, 3, 4, 5]          # n-vs-n evaluation scenarios

if SMOKE:
    HP = dict(low_envs=4, low_rollout=40, low_minibatch=40, low_epochs=2,
              cmd_envs=3, cmd_rollout=20, cmd_minibatch=40, cmd_epochs=2)
    STAGE_BUDGET = {k: (160 if not k.startswith('commander') else 60) for k in STAGE_BUDGET}
    EVAL_EPISODES = 3
    CKPT_EVERY_MIN = 0.01

HORIZON_LOW = {1: 150, 2: 200, 3: 300, 4: 350, 5: 400}
if PAPER_HORIZONS:
    HORIZON_LOW = {1: 200, 2: 250, 3: 300, 4: 350, 5: 400}
HORIZON_HL = 500
MAP_LOW, MAP_HL = 0.3, 0.5         # degrees (about 33 km / 55 km)
HIT_PROB = (0.70, 0.85) if PAPER_HIT_PROB else (0.75, 0.90)

CONFIG = dict(SMOKE=SMOKE, PRESET=PRESET, BUDGET=BUDGET, HP=HP, STAGE_BUDGET=STAGE_BUDGET, SEED=SEED,
              RLLIB_FAITHFUL=RLLIB_FAITHFUL, PAPER_PPO=PAPER_PPO, PAPER_HORIZONS=PAPER_HORIZONS,
              PAPER_HIT_PROB=PAPER_HIT_PROB, PAPER_HL_OOB=PAPER_HL_OOB, PAPER_L5_POOL=PAPER_L5_POOL,
              ESC_DIST_REW=ESC_DIST_REW, FRIENDLY_KILL=FRIENDLY_KILL, FRIENDLY_PUNISH=FRIENDLY_PUNISH,
              HIER_OPP_FIGHT_RATIO=HIER_OPP_FIGHT_RATIO, HIER_ACTION_ASSESS=HIER_ACTION_ASSESS,
              HL_OPP_START_SPEED0=HL_OPP_START_SPEED0, PPO_LOW=PPO_LOW, PPO_CMD=PPO_CMD,
              EVAL_EPISODES=EVAL_EPISODES, EXPERIMENTS=EXPERIMENTS)
with open(os.path.join(OUT_DIR, 'config.json'), 'w') as _f:
    json.dump(CONFIG, _f, indent=1)
print(json.dumps(CONFIG, indent=1))

## 3. The JAX environment

One `Env` class covers both original environments:

* `kind='low'` corresponds to `LowLevelEnv` (2 vs 2, 30 km map). It runs fight or escape mode and curriculum levels 1–5:
  * L1 static, L2 random and L3 scripted opponents;
  * L4 and L5 fictitious self-play opponents driven by **frozen policy parameters** (deterministic argmax, like `_policy_actions`).
* `kind='high'` corresponds to `HighLevelEnv` (N vs M up to 5 vs 5, 50 km map). One `step` is a commander step:
  * opponent fight/escape assignment;
  * then up to 16 low-level ticks, in which every aircraft is driven by frozen low-level fight or escape policies;
  * the commander step exits early on any kill/out-of-bounds event, or on a close engagement after tick 12.

Units are fixed slots `0..N-1` (agents, blue) and `N..N+M-1` (opponents, red); dead units are masked
(`alive`). Every function is pure and written for a single env; batching is `jax.vmap`.

Faithfulness notes (see `docs/spec_original_env.md`), all reproduced:
* **Tick order:**
  * aircraft update in id order, then rockets in launch order;
  * units killed earlier in the same tick still act (fire, steer, move);
  * the cannon kills immediately, with a Bernoulli p_hit/burst per target per tick;
  * the cannon cone uses the post-turn heading and the pre-move position.
* **Weapon quirks:**
  * rockets are unguided, with multiplicative heading noise clip(h·U(0.95,1.05), 0, 359);
  * the radar cone is [h, h+120];
  * the cooldown is set even when a launch fails;
  * scripted opponents never count their cooldown down;
  * every rocket's proximity fuse is hard-coded to aircraft id 2;
  * a rocket hit is checked before the rocket moves.
* **Observation quirks:**
  * headings are normalized by 359;
  * observation and reward geometry is planar in degrees, weapon geometry is geodesic (local ellipsoidal flat-earth);
  * FSP opponents observe the agents' *current-step* shot flags;
  * the commander's low-level observations use distances frozen at the start of the commander step;
  * an escaping aircraft's missile targets the farthest opponent.
* **Reward and episode quirks:**
  * kill rewards earned by an agent that died earlier are dropped;
  * dead agents keep emitting zero observations;
  * the time limit is a true terminal.

In [ ]:
KN2MS = 0.514444
_WGS_A = 6378137.0
_WGS_F = 1.0 / 298.257223563
_WGS_E2 = _WGS_F * (2.0 - _WGS_F)
LAT0 = 5.0                    # map origin (lat 5, lon 7); positions are stored as offsets in degrees
D2R = math.pi / 180.0
# rocket speed profile interp1d([0,10,20,30],[500,2000,1400,600],'quadratic') at integer life 0..10 s
ROCKET_SPEED = np.array([500.0, 758.75, 993.3333333, 1203.75, 1390.0, 1552.0833333, 1690.0, 1803.75,
                         1893.3333333, 1958.75, 2000.0], np.float32)
SPLITS1, SPLITS2 = (13, 9, 2, 2), (13, 9, 2)        # MultiDiscrete action spaces of AC1 / AC2
OBS_FIGHT = (26, 24)
OBS_ESC = (30, 29)
OBS_CMD = 34


def _radii(lat_abs):
    s = jnp.sin(lat_abs * D2R)
    w = jnp.sqrt(1.0 - _WGS_E2 * s * s)
    return _WGS_A * (1.0 - _WGS_E2) / w ** 3, _WGS_A / w        # meridional M, normal N


def geo_inv(lat1, lon1, lat2, lon2):
    """Geodesic distance [km] and initial bearing [deg, 0..360) (local ellipsoidal approximation,
    < 0.2 m / < 0.001 deg error on these maps vs geographiclib)."""
    lm = LAT0 + 0.5 * (lat1 + lat2)
    M, N = _radii(lm)
    dn = (lat2 - lat1) * D2R * M
    de = (lon2 - lon1) * D2R * N * jnp.cos(lm * D2R)
    dist = jnp.sqrt(dn * dn + de * de) / 1000.0
    brg = jnp.degrees(jnp.arctan2(de, dn)) - (lon2 - lon1) * jnp.sin(lm * D2R) / 2.0
    return dist, jnp.mod(brg, 360.0)


def geo_move(lat, lon, hdg, dist_m):
    """Geodesic direct step (midpoint flat-earth, < 1 cm error per 1 km step)."""
    c, s = jnp.cos(hdg * D2R), jnp.sin(hdg * D2R)
    M, _ = _radii(LAT0 + lat)
    lm = LAT0 + lat + 0.5 * dist_m * c / M / D2R
    M, N = _radii(lm)
    return lat + dist_m * c / M / D2R, lon + dist_m * s / (N * jnp.cos(lm * D2R)) / D2R


def shd(actual, desired):
    """warsim signed_heading_diff."""
    d = desired - actual
    d = jnp.where(d < -180.0, d + 360.0, d)
    return jnp.where(d > 180.0, d - 360.0, d)


def tree_where(c, a, b):
    return jax.tree_util.tree_map(lambda x, y: jnp.where(jnp.reshape(c, c.shape + (1,) * (x.ndim - c.ndim)), x, y), a, b)


def _uni(k, lo, hi, shape=()):
    return jax.random.uniform(k, shape, minval=lo, maxval=hi)


def _randint(k, lo, hi, shape=()):          # inclusive bounds like python random.randint
    return jax.random.randint(k, shape, lo, hi + 1)


class Env:
    """kind='low': LowLevelEnv (2v2). kind='high': HighLevelEnv (NvM, commander steps)."""

    def __init__(self, kind='low', n=2, m=2, level=3, mode='fight', opp_pool=None, record=False):
        self.kind, self.hl = kind, kind == 'high'
        self.N, self.M, self.T, self.K = n, m, n + m, max(n, m)
        self.level, self.mode, self.record = level, mode, record
        self.ms = MAP_HL if self.hl else MAP_LOW
        self.horizon = HORIZON_HL if self.hl else HORIZON_LOW[level]
        self.dnorm = math.sqrt(2 * self.ms ** 2)
        self.team = np.array([0] * n + [1] * m)
        self.is_ag = jnp.asarray(self.team == 0)
        if not self.hl:
            assert n == 2 and m == 2, 'the original low-level env is 2 vs 2 (fixed partners 1<->2, 3<->4)'
            self.partner = jnp.array([1, 0, 3, 2])
            if level == 4:
                opp_pool = opp_pool or ('fight',)
            elif level == 5:
                opp_pool = opp_pool or ((('fight',) * 4) if PAPER_L5_POOL else ('fight', 'fight', 'escape'))
            self.opp_pool = tuple(opp_pool or ())
        self.obs_dim = OBS_CMD if self.hl else (26 if mode == 'fight' else 30)
        self.p_tick = (HIT_PROB[0] / 5.0, HIT_PROB[1] / 3.0)
        self.oob_pen = (-5.0 if PAPER_HL_OOB else -2.0) if self.hl else -5.0

    # ------------------------------------------------------------------ reset
    def reset(self, key):
        N, M, T = self.N, self.M, self.T
        ks = jax.random.split(key, 8)
        r = _randint(ks[0], 1, 2)
        team = jnp.asarray(self.team)
        idx = jnp.asarray(np.r_[np.arange(N), np.arange(M)]).astype(jnp.float32)
        u = jax.random.uniform(ks[1], (T, 2))
        ag = team == 0
        left = jnp.where(ag, r == 1, r == 2)          # this unit spawns on the west side
        L = self.level
        if self.hl:
            K = jnp.where(ag, float(N), float(M))
            lon = jnp.where(left, 0.07 + 0.15 * u[:, 0], 0.28 + 0.15 * u[:, 0])
            lat = 0.07 + idx * 0.4 / K + 0.05 * u[:, 1]
            hdg = _randint(ks[2], 0, 359, (T,)).astype(jnp.float32)
        else:
            if L == 1:
                lon = jnp.where(left, 0.12 + 0.02 * u[:, 0], 0.16 + 0.01 * u[:, 0])
                lat = 0.10 + 0.1 * idx + 0.01 * u[:, 1]
                h_ag = jnp.where(r == 1, _randint(ks[2], 30, 150, (T,)), _randint(ks[3], 200, 330, (T,)))
                hdg = jnp.where(ag, h_ag, 0).astype(jnp.float32)
            elif L == 2:
                lon = jnp.where(left, 0.08 + 0.05 * u[:, 0], 0.18 + 0.05 * u[:, 0])
                lat = 0.08 + 0.1 * idx + 0.05 * u[:, 1]
                h_ag = jnp.where(r == 1, _randint(ks[2], 0, 180, (T,)), _randint(ks[3], 180, 359, (T,)))
                hdg = jnp.where(ag, h_ag, _randint(ks[4], 0, 359, (T,))).astype(jnp.float32)
            else:
                lon = jnp.where(left, 0.07 + 0.05 * u[:, 0], 0.18 + 0.05 * u[:, 0])
                lat = 0.09 + 0.1 * idx + 0.03 * u[:, 1]
                h_ag = jnp.where(r == 1, _randint(ks[2], 0, 270, (T,)), _randint(ks[3], 90, 359, (T,)))
                hdg = jnp.where(ag, h_ag, _randint(ks[4], 0, 359, (T,))).astype(jnp.float32)
        slot = jnp.asarray(np.r_[np.arange(N), np.arange(M)])
        ac1 = jnp.where(slot == 0, True, jnp.where(slot == 1, False, jax.random.bernoulli(ks[5], 0.5, (T,))))
        opp_slow = (~ag) & ((L <= 2) if not self.hl else HL_OPP_START_SPEED0)
        spd = jnp.where(opp_slow, 0.0, 100.0)
        if self.hl:
            can = jnp.full((T,), 300.0); mis = jnp.where(ac1, 8.0, 0.0)
        elif L == 5:
            can = jnp.full((T,), 300.0); mis = jnp.where(ac1, 6.0, 0.0)
        else:
            can = jnp.where(ag, 200.0, 400.0); mis = jnp.where(ac1, jnp.where(ag, 5.0, 8.0), 0.0)
        f0, i0, b0 = jnp.zeros((T,), jnp.float32), jnp.zeros((T,), jnp.int32), jnp.zeros((T,), bool)
        s = dict(lat=lat.astype(jnp.float32), lon=lon.astype(jnp.float32), hdg=hdg, nhdg=hdg, spd=spd, nspd=spd,
                 ac1=ac1, alive=jnp.ones((T,), bool), can_rem=can, can_max=can, burst=f0,
                 mis_rem=mis, mis_max=mis, has_mis=b0, mwait=i0,
                 r_alive=b0, r_lat=f0, r_lon=f0, r_hdg=f0, r_nhdg=f0, r_spd=f0, r_tgt=i0, r_t0=i0, r_seq=i0,
                 seq_ctr=jnp.int32(0), t=jnp.int32(0), steps=jnp.int32(0),
                 esc_flag=jnp.array(False), esc_timer=jnp.int32(0), tgt=jnp.full((T,), -1, jnp.int32),
                 opp_k=jnp.int32(0), ca=jnp.zeros((T,), jnp.int32),
                 lst=jnp.full((T, self.K), -1, jnp.int32), lst_dn=jnp.zeros((T, self.K)), lst_n=i0,
                 # episode statistics for evaluation: kills/deaths/friendly kills per agent aircraft type
                 st_k=jnp.zeros(2), st_d=jnp.zeros(2), st_fk=jnp.zeros(2), st_ag_by_opp=jnp.float32(0),
                 st_opp_by_ag=jnp.float32(0), st_oob=jnp.float32(0))
        if not self.hl and L == 5:
            s['opp_k'] = jax.random.randint(ks[6], (), 0, len(self.opp_pool))
        if self.hl:
            obs, s = self._cmd_obs(s)
        else:
            obs, s = self._low_obs(s)
        return s, obs

    # ------------------------------------------------------------------ geometry & observations
    def _planar(self, s):
        dx = s['lon'][None, :] - s['lon'][:, None]
        dy = s['lat'][None, :] - s['lat'][:, None]
        dd = jnp.sqrt(dx * dx + dy * dy)                                    # [i,j] planar distance (deg)
        hr = s['hdg'] * D2R
        ux, uy = jnp.sin(hr), jnp.cos(hr)
        dot = ux[:, None] * dx + uy[:, None] * dy
        foc = jnp.degrees(jnp.arccos(jnp.clip(dot / (dd + 1e-10), -1.0, 1.0)))   # [i,j] ATA of i towards j
        hd = jnp.degrees(jnp.arccos(jnp.clip((ux[:, None] * ux[None, :] + uy[:, None] * uy[None, :]) / (1.0 + 1e-10),
                                             -1.0, 1.0))) / 180.0
        return dd, foc, jnp.clip(hd, 0.0, 1.0)

    def _own(self, s):
        latr = jnp.clip(s['lat'] / self.ms, 0.0, 1.0)
        lonr = jnp.clip(s['lon'] / self.ms, 0.0, 1.0)
        spdn = jnp.clip(s['spd'] / jnp.where(s['ac1'], 900.0, 600.0), 0.0, 1.0)
        hn = jnp.clip(jnp.mod(s['hdg'], 359.0) / 359.0, 0.0, 1.0)
        return latr, lonr, spdn, hn

    @staticmethod
    def _shot(s):
        return ((s['burst'] > 0) | s['has_mis']).astype(jnp.float32)

    def _sorted_enemies(self, s, dd):
        team = jnp.asarray(self.team)
        enemy = (team[:, None] != team[None, :]) & s['alive'][None, :]
        dm = jnp.where(enemy, dd, jnp.inf)
        order = jnp.argsort(dm, axis=1, stable=True)[:, :self.K]
        n = jnp.sum(enemy, axis=1) * s['alive']
        kk = jnp.arange(self.K)[None, :]
        lst = jnp.where(kk < n[:, None], order, -1).astype(jnp.int32)
        lst_dn = jnp.take_along_axis(dd, jnp.maximum(lst, 0), axis=1) / self.dnorm
        return lst, jnp.where(lst >= 0, lst_dn, 0.0), n.astype(jnp.int32)

    def _sorted_friends(self, s, dd):
        team = jnp.asarray(self.team)
        fr = (team[:, None] == team[None, :]) & s['alive'][None, :] & ~jnp.eye(self.T, dtype=bool)
        dm = jnp.where(fr, dd, jnp.inf)
        order = jnp.argsort(dm, axis=1, stable=True)
        n = jnp.sum(fr, axis=1)
        return jnp.where(jnp.arange(self.T)[None, :] < n[:, None], order, -1)

    def _fri_block(self, s, P, fri):
        dd, foc, _ = P
        ar = jnp.arange(self.T)
        f = jnp.maximum(fri, 0)
        latr, lonr, _, _ = self._own(s)
        ok = (fri >= 0) & s['alive'][f]
        blk = jnp.stack([latr[f], lonr[f], jnp.clip(foc[ar, f] / 180, 0, 1), jnp.clip(foc[f, ar] / 180, 0, 1),
                         dd[ar, f] / self.dnorm], -1)
        return jnp.where(ok[:, None], blk, 0.0)

    def _fight_rows(self, s, P, tgt, tdn, fri, oflag):
        """fight_state_values for every unit -> [T, 26]; AC2 rows use the first 24 entries."""
        dd, foc, hd = P
        ar = jnp.arange(self.T)
        j = jnp.maximum(tgt, 0)
        latr, lonr, spdn, hn = self._own(s)
        fij, fji = jnp.clip(foc[ar, j] / 180, 0, 1), jnp.clip(foc[j, ar] / 180, 0, 1)
        can = jnp.clip(s['can_rem'] / s['can_max'], 0, 1)
        mis = jnp.clip(s['mis_rem'] / jnp.maximum(s['mis_max'], 1.0), 0, 1)
        base = jnp.stack([latr, lonr, spdn, hn, fij, 1.0 - fji, hd[ar, j], tdn, can], -1)
        opp = jnp.stack([latr[j], lonr[j], spdn[j], hn[j], hd[j, ar], fji, 1.0 - fij, tdn, oflag[j]], -1)
        fr = self._fri_block(s, P, fri)
        own_shot = self._shot(s)
        r1 = jnp.concatenate([base, mis[:, None], (s['mwait'] == 0).astype(jnp.float32)[:, None],
                              own_shot[:, None], opp, fr], -1)
        r2 = jnp.concatenate([base, (s['burst'] > 0).astype(jnp.float32)[:, None], opp, fr,
                              jnp.zeros((self.T, 2))], -1)
        row = jnp.where(s['ac1'][:, None], r1, r2)
        return jnp.where((s['alive'] & (tgt >= 0))[:, None], row, 0.0)

    def _esc_rows(self, s, P, e, edn, en, fri, oflag):
        """esc_state_values -> [T, 30]; e/edn: [T, >=2] nearest-first enemy ids / normalized dists, en: count."""
        dd, foc, hd = P
        ar = jnp.arange(self.T)
        latr, lonr, spdn, hn = self._own(s)
        can = jnp.clip(s['can_rem'] / s['can_max'], 0, 1)
        mis = jnp.clip(s['mis_rem'] / jnp.maximum(s['mis_max'], 1.0), 0, 1)
        shot = self._shot(s)
        blks = []
        for k in range(2):
            o = jnp.maximum(e[:, k], 0)
            b = jnp.stack([latr[o], lonr[o], spdn[o], hn[o], hd[o, ar], jnp.clip(foc[ar, o] / 180, 0, 1),
                           jnp.clip(foc[o, ar] / 180, 0, 1), edn[:, k], oflag[o]], -1)
            blks.append(jnp.where((en > k)[:, None], b, 0.0))
        fr = self._fri_block(s, P, fri)
        r1 = jnp.concatenate([jnp.stack([latr, lonr, spdn, hn, can, mis, shot], -1)] + blks + [fr], -1)
        r2 = jnp.concatenate([jnp.stack([latr, lonr, spdn, hn, can, shot], -1)] + blks + [fr, jnp.zeros((self.T, 1))], -1)
        row = jnp.where(s['ac1'][:, None], r1, r2)
        return jnp.where((s['alive'] & (en > 0))[:, None], row, 0.0)

    def _low_obs(self, s):
        """LowLevelEnv.state(): nearest opponent, fixed partner; also stores opp_to_attack (s['tgt'])."""
        P = self._planar(s)
        lst, lst_dn, n = self._sorted_enemies(s, P[0])
        flags = self._shot(s)
        tgt = jnp.where(n > 0, lst[:, 0], -1)
        if self.mode == 'fight':
            rows = self._fight_rows(s, P, tgt, lst_dn[:, 0], self.partner, flags)
        else:
            rows = self._esc_rows(s, P, lst, lst_dn, n, self.partner, flags)
        s = dict(s)
        s['tgt'] = s['tgt'].at[:self.N].set(tgt[:self.N])
        return rows[:self.N], s

    def _cmd_obs(self, s):
        """HighLevelEnv.state(): 34-dim commander obs for agents; stores the sorted enemy lists of all units."""
        dd, foc, hd = P = self._planar(s)
        lst, lst_dn, n = self._sorted_enemies(s, dd)
        fr = self._sorted_friends(s, dd)
        ar = jnp.arange(self.T)
        latr, lonr, spdn, hn = self._own(s)
        parts = [jnp.stack([latr, lonr, spdn, hn], -1)]
        for k in range(2):
            o = jnp.maximum(lst[:, k], 0)
            fio, foi = jnp.clip(foc[ar, o] / 180, 0, 1), jnp.clip(foc[o, ar] / 180, 0, 1)
            b = jnp.stack([latr[o], lonr[o], spdn[o], hn[o], hd[o, ar], fio, foi, 1 - fio, 1 - foi, lst_dn[:, k]], -1)
            parts.append(jnp.where((n > k)[:, None], b, 0.0))
        for k in range(2):
            parts.append(self._fri_block(s, P, fr[:, k]) if k < self.T else jnp.zeros((self.T, 5)))
        rows = jnp.concatenate(parts, -1)
        rows = jnp.where((s['alive'] & (n > 0))[:, None], rows, 0.0)
        s = dict(s)
        s.update(lst=lst, lst_dn=lst_dn, lst_n=n)
        return rows[:self.N], s

    # ------------------------------------------------------------------ actions
    def _fire_missile(self, s, try_mask, tgt):
        """Rafale.fire_missile for all units in try_mask (radar cone [h, h+120] quirk; launch in id order)."""
        j = jnp.maximum(tgt, 0)
        d_km, brg = geo_inv(s['lat'], s['lon'], s['lat'][j], s['lon'][j])
        radar = jnp.floor(jnp.abs(shd(jnp.mod(s['hdg'] + 60.0, 360.0), brg))) <= 60.0
        ok = try_mask & (tgt >= 0) & ~s['has_mis'] & (s['mis_rem'] > 0) & (d_km <= 111.0) & radar
        seq = s['seq_ctr'] + jnp.cumsum(ok.astype(jnp.int32)) - 1
        s = dict(s)
        s.update(r_alive=s['r_alive'] | ok, r_lat=jnp.where(ok, s['lat'], s['r_lat']),
                 r_lon=jnp.where(ok, s['lon'], s['r_lon']), r_hdg=jnp.where(ok, s['hdg'], s['r_hdg']),
                 r_nhdg=jnp.where(ok, s['hdg'], s['r_nhdg']), r_spd=jnp.where(ok, 500.0, s['r_spd']),
                 r_tgt=jnp.where(ok, j, s['r_tgt']), r_t0=jnp.where(ok, s['t'], s['r_t0']),
                 r_seq=jnp.where(ok, seq, s['r_seq']), seq_ctr=s['seq_ctr'] + jnp.sum(ok),
                 has_mis=s['has_mis'] | ok, mis_rem=s['mis_rem'] - ok)
        return s

    def _base_action(self, s, mask, a, tgt, key):
        """_take_base_action for every unit in `mask` (a: [T,4] ints). Returns state, escape penalties [T]."""
        maxspd = jnp.where(s['ac1'], 900.0, 600.0)
        s = dict(s)
        s['nhdg'] = jnp.where(mask, jnp.mod(s['hdg'] + (a[:, 0] - 6) * 15.0, 360.0), s['nhdg'])
        s['nspd'] = jnp.where(mask, 100.0 + ((maxspd - 100.0) / 8.0) * a[:, 1], s['nspd'])
        fire_c = mask & (a[:, 2] == 1) & (s['can_rem'] > 0)
        pen = jnp.where(fire_c & (s['can_rem'] < 90), -0.1, 0.0)
        s['burst'] = jnp.where(fire_c, jnp.minimum(s['can_rem'], jnp.where(s['ac1'], 5.0, 3.0)), s['burst'])
        try_m = mask & s['ac1'] & (a[:, 3] == 1) & (tgt >= 0) & (s['mis_rem'] > 0) & ~s['has_mis'] & (s['mwait'] == 0)
        s = self._fire_missile(s, try_m, tgt)
        wait = _randint(key, 8, 12, (self.T,)) if self.hl else _randint(key, 7, 17, (self.T,))
        s['mwait'] = jnp.where(try_m, wait, s['mwait'])           # cooldown set even if the launch failed
        pen = pen + jnp.where(try_m & (s['mis_rem'] < 3), -0.1, 0.0)
        s['mwait'] = jnp.where(mask & (s['mwait'] > 0) & ~s['has_mis'], s['mwait'] - 1, s['mwait'])
        return s, pen

    def _scripted_opps(self, s, key):
        """Level 1-3 opponents (env_hetero.__opp_level1/2/3), processed in id order with the shared L3 escape state."""
        N, T, L = self.N, self.T, self.level
        dd, foc, _ = self._planar(s)
        lst, lst_dn, n = self._sorted_enemies(s, dd)
        near, ndn, has = lst[:, 0], lst_dn[:, 0], n > 0
        latr, lonr, _, _ = self._own(s)
        keys = jax.random.split(key, T)
        for i in range(N, T):
            k = jax.random.split(keys[i], 12)
            alive = s['alive'][i]
            onehot = jnp.arange(T) == i
            s = dict(s)
            if L in (1, 2):
                if L == 2:
                    s['burst'] = jnp.where(onehot & alive, jnp.minimum(s['can_rem'], jnp.where(s['ac1'], 5.0, 3.0)),
                                           s['burst'])
                    turn = (s['steps'] <= 5) | (jnp.mod(s['steps'], _randint(k[0], 35, 45)) <= 5)
                    sign = jnp.where(_randint(k[1], 0, 1) == 0, 90.0, -90.0)
                    nh = jnp.mod(s['hdg'][i] + sign, 360.0)
                    ns = 100.0 + _randint(k[2], 0, 4) * 75.0
                    s['nhdg'] = jnp.where(onehot & alive & turn, nh, s['nhdg'])
                    s['nspd'] = jnp.where(onehot & alive & turn, ns, s['nspd'])
                cond = (alive & ~s['has_mis'][i] & (jnp.mod(s['steps'], 40) < 3) & (_randint(k[3], 0, 1) == 1)
                        & (s['mwait'][i] == 0) & s['ac1'][i] & has[i])
                s = self._fire_missile(s, onehot & cond, jnp.where(onehot, near, -1))
                s['mwait'] = jnp.where(onehot & cond, 5, s['mwait'])
            else:  # level 3
                roll = alive & (jnp.mod(s['steps'], 60) == 0) & ~s['esc_flag']
                coin = _randint(k[0], 0, 1) == 1
                flag = jnp.where(roll, coin, s['esc_flag'])
                timer = jnp.where(roll & coin, jnp.floor(_uni(k[1], 20.0, 30.0)).astype(jnp.int32), s['esc_timer'])
                # escaping opponent: diagonal flight back to the map centre
                y, x = latr[i], lonr[i]
                lo = jnp.where(y < 0.5, jnp.where(x < 0.5, 30.0, 300.0), jnp.where(x < 0.5, 120.0, 210.0))
                h_esc = jnp.floor(_uni(k[2], lo, lo + 30.0))
                sp_esc = jnp.floor(_uni(k[3], 300.0, 600.0))
                fire_esc = _randint(k[4], 0, 1) == 1
                # hard-coded pursuit of the nearest agent
                ag = jnp.maximum(near[i], 0)
                hdg = s['hdg'][i]
                x0, y0 = s['lon'][i], s['lat'][i]
                x1 = x0 + jnp.round(jnp.sin(jnp.mod(hdg, 360.0) * D2R), 3)
                y1 = y0 + jnp.round(jnp.cos(jnp.mod(hdg, 360.0) * D2R), 3)
                val = (x1 - x0) * (s['lat'][ag] - y0) - (s['lon'][ag] - x0) * (y1 - y0)
                sign = jnp.where(val < 0, 1.0, -1.0)
                rr = _uni(k[5], 0.7, 1.3)
                fo = foc[i, ag]
                d = ndn[i]
                sp_h = jnp.floor(_uni(k[6], 100.0, 400.0))
                h_h = jnp.where(has[i] & (d > 0.008) & (fo > 4), jnp.mod(hdg + rr * sign * fo, 360.0), hdg)
                sp_far = jnp.where(fo < 30, jnp.floor(_uni(k[7], 500.0, 800.0)), jnp.floor(_uni(k[8], 100.0, 500.0)))
                sp_h = jnp.where(has[i] & (d > 0.05), sp_far, sp_h)
                sp_h = jnp.where(s['ac1'][i], sp_h, jnp.minimum(sp_h, 600.0))
                fire_h = has[i] & (d < 0.03) & (fo < 10)
                firem_h = has[i] & (d < 0.09) & (fo < 5)
                esc = flag
                heading = jnp.where(esc, h_esc, h_h)
                speed = jnp.where(esc, sp_esc, sp_h)
                fire = jnp.where(esc, fire_esc, fire_h)
                firem = jnp.where(esc, False, firem_h)
                timer = jnp.where(esc, timer - 1, timer)
                flag = jnp.where(esc & (timer <= 0), False, flag)
                s['esc_flag'] = jnp.where(alive, flag, s['esc_flag'])
                s['esc_timer'] = jnp.where(alive, timer, s['esc_timer'])
                m = onehot & alive
                s['nhdg'] = jnp.where(m, heading, s['nhdg'])
                s['nspd'] = jnp.where(m, speed, s['nspd'])
                s['burst'] = jnp.where(m & fire, jnp.minimum(s['can_rem'], jnp.where(s['ac1'], 5.0, 3.0)), s['burst'])
                cond = alive & firem & ~s['has_mis'][i] & (s['mwait'][i] == 0) & s['ac1'][i]
                s = self._fire_missile(s, onehot & cond, jnp.where(onehot, near, -1))
                s['mwait'] = jnp.where(onehot & cond, 10, s['mwait'])
        return s

    def _fsp_opps(self, s, key, opp_params):
        """Level 4/5 opponents: frozen policies (argmax). They observe the agents' current-step shot flags."""
        P = self._planar(s)
        lst, lst_dn, n = self._sorted_enemies(s, P[0])
        flags = self._shot(s)
        tgt = jnp.where(n > 0, lst[:, 0], -1)
        obs_f = self._fight_rows(s, P, tgt, lst_dn[:, 0], self.partner, flags)
        obs_e = self._esc_rows(s, P, lst, lst_dn, n, self.partner, flags)
        acts = [frozen_policy_actions(kind, p, obs_f if kind == 'fight' else obs_e, s['ac1'])
                for kind, p in zip(self.opp_pool, opp_params)]
        a = acts[0]
        for k in range(1, len(acts)):
            a = jnp.where(s['opp_k'] == k, acts[k], a)
        s = dict(s)
        s['tgt'] = s['tgt'].at[self.N:].set(tgt[self.N:])
        mask = s['alive'] & ~self.is_ag
        s, _ = self._base_action(s, mask, a, s['tgt'], key)
        return s

    # ------------------------------------------------------------------ simulator tick
    def _tick(self, s, key):
        T = self.T
        team = jnp.asarray(self.team)
        alive0, ralive0 = s['alive'], s['r_alive']
        ka, kb = jax.random.split(key)
        u_hit = jax.random.uniform(ka, (T, T))
        u_mis = _uni(kb, 0.95, 1.05, (T,))
        killer = jnp.full((T,), -1, jnp.int32)
        kind = jnp.zeros((T,), jnp.int32)          # 0 cannon, 1 rocket
        ar = jnp.arange(T)

        def ac_body(i, c):
            s, killer, kind = c
            upd = alive0[i]
            ac1 = s['ac1'][i]
            h, nh = s['hdg'][i], s['nhdg'][i]
            maxd = jnp.where(ac1, 5.0, 3.5)
            dl = shd(h, nh)
            h2 = jnp.where(jnp.abs(dl) <= maxd, nh, jnp.mod(h + jnp.where(dl >= 0, maxd, -maxd), 360.0))
            h2 = jnp.where(h != nh, h2, h)
            sp, ns = s['spd'][i], s['nspd'][i]
            maxa = jnp.where(ac1, 35.0, 28.0)
            ds = ns - sp
            sp2 = jnp.where(jnp.abs(ds) <= maxa, ns, sp + jnp.where(ds >= 0, maxa, -maxa))
            sp2 = jnp.where(sp != ns, sp2, sp)
            # cannon
            burst, rem = s['burst'][i], s['can_rem'][i]
            firing = upd & (burst > 0)
            d_km, brg = geo_inv(s['lat'][i], s['lon'][i], s['lat'], s['lon'])
            cone = (d_km < jnp.where(ac1, 2.0, 4.5)) & (jnp.abs(shd(h2, brg)) <= jnp.where(ac1, 5.0, 3.5))
            ff = (team != team[i]) | FRIENDLY_KILL
            p = jnp.where(ac1, self.p_tick[0], self.p_tick[1])
            hit = firing & s['alive'] & (ar != i) & cone & ff & (u_hit[i] < p)
            killer = jnp.where(hit, i, killer)
            kind = jnp.where(hit, 0, kind)
            # missile steering (unguided: multiplicative heading noise)
            hm, rok = s['has_mis'][i], s['r_alive'][i]
            steer = upd & hm & rok
            r_nh = jnp.clip(s['r_hdg'][i] * u_mis[i], 0.0, 359.0)
            # move
            la2, lo2 = geo_move(s['lat'][i], s['lon'][i], h2, sp2 * KN2MS)
            mv = upd & (sp2 > 0)
            s = dict(s)
            s['alive'] = s['alive'] & ~hit
            s['hdg'] = s['hdg'].at[i].set(jnp.where(upd, h2, h))
            s['spd'] = s['spd'].at[i].set(jnp.where(upd, sp2, sp))
            s['burst'] = s['burst'].at[i].set(jnp.where(firing, jnp.maximum(burst - 1.0, 0.0), burst))
            s['can_rem'] = s['can_rem'].at[i].set(jnp.where(firing, jnp.maximum(rem - 1.0, 0.0), rem))
            s['has_mis'] = s['has_mis'].at[i].set(jnp.where(upd & hm & ~rok, False, hm))
            s['r_nhdg'] = s['r_nhdg'].at[i].set(jnp.where(steer, r_nh, s['r_nhdg'][i]))
            s['lat'] = s['lat'].at[i].set(jnp.where(mv, la2, s['lat'][i]))
            s['lon'] = s['lon'].at[i].set(jnp.where(mv, lo2, s['lon'][i]))
            return s, killer, kind

        s, killer, kind = jax.lax.fori_loop(0, T, ac_body, (s, killer, kind))
        order = jnp.argsort(jnp.where(ralive0, s['r_seq'], jnp.iinfo(jnp.int32).max))

        def r_body(k, c):
            s, killer, kind = c
            j = order[k]
            upd = ralive0[j]
            tg = s['r_tgt'][j]
            d_t, _ = geo_inv(s['r_lat'][j], s['r_lon'][j], s['lat'][tg], s['lon'][tg])
            hit_t = upd & (d_t < 1.0) & s['alive'][tg]
            fid = jnp.where(j == 1, 0, 1)          # proximity fuse hard-coded to aircraft id 2 (bug)
            d_f, _ = geo_inv(s['r_lat'][j], s['r_lon'][j], s['lat'][fid], s['lon'][fid])
            hit_f = upd & ~hit_t & FRIENDLY_KILL & s['alive'][fid] & (d_f < 1.0)
            life = s['t'] - s['r_t0'][j]
            eol = upd & ~hit_t & ~hit_f & (life > 10)
            cont = upd & ~hit_t & ~hit_f & ~eol
            s = dict(s)
            alive = s['alive'].at[tg].set(jnp.where(hit_t, False, s['alive'][tg]))
            s['alive'] = alive.at[fid].set(jnp.where(hit_f, False, alive[fid]))
            killer = killer.at[tg].set(jnp.where(hit_t, j, killer[tg]))
            kind = kind.at[tg].set(jnp.where(hit_t, 1, kind[tg]))
            killer = killer.at[fid].set(jnp.where(hit_f, j, killer[fid]))
            kind = kind.at[fid].set(jnp.where(hit_f, 1, kind[fid]))
            rh, rnh = s['r_hdg'][j], s['r_nhdg'][j]
            dl = shd(rh, rnh)
            rh2 = jnp.where(rh != rnh, jnp.where(jnp.abs(dl) <= 10.0, rnh, rh + jnp.where(dl >= 0, 10.0, -10.0)), rh)
            rs = jnp.asarray(ROCKET_SPEED)[jnp.clip(life, 0, 10)]
            la2, lo2 = geo_move(s['r_lat'][j], s['r_lon'][j], rh2, rs * KN2MS)
            s['r_alive'] = s['r_alive'].at[j].set(jnp.where(hit_t | hit_f | eol, False, s['r_alive'][j]))
            s['r_hdg'] = s['r_hdg'].at[j].set(jnp.where(cont, rh2, rh))
            s['r_spd'] = s['r_spd'].at[j].set(jnp.where(cont, rs, s['r_spd'][j]))
            s['r_lat'] = s['r_lat'].at[j].set(jnp.where(cont, la2, s['r_lat'][j]))
            s['r_lon'] = s['r_lon'].at[j].set(jnp.where(cont, lo2, s['r_lon'][j]))
            return s, killer, kind

        s, killer, kind = jax.lax.fori_loop(0, T, r_body, (s, killer, kind))
        s['t'] = s['t'] + 1
        return s, killer, kind

    # ------------------------------------------------------------------ rewards & bookkeeping
    def _combat(self, s, killer, kind):
        """Out-of-boundary removal + per-agent combat outcome terms. Returns state and outcome dict."""
        N = self.N
        inb = (s['lat'] >= 0) & (s['lat'] <= self.ms) & (s['lon'] >= 0) & (s['lon'] <= self.ms)
        oob = s['alive'] & ~inb
        s = dict(s)
        s['alive'] = s['alive'] & ~oob
        is_ag = self.is_ag
        by_ag = (killer >= 0) & (killer < N)
        by_op = killer >= N
        ka = killer[None, :] == jnp.arange(N)[:, None]              # [agent, victim]
        kills_opp = ka & ~is_ag[None, :]
        kills_fr = ka & is_ag[None, :]
        t1 = s['ac1'][:N].astype(jnp.float32)
        tw = jnp.stack([t1, 1 - t1], -1)                              # agent type one-hot [N,2]
        dead_ag = (by_ag | by_op)[:N] | oob[:N]
        s['st_k'] = s['st_k'] + jnp.sum(jnp.sum(kills_opp, 1)[:, None] * tw, 0)
        s['st_fk'] = s['st_fk'] + jnp.sum(jnp.sum(kills_fr, 1)[:, None] * tw, 0)
        s['st_d'] = s['st_d'] + jnp.sum(dead_ag[:, None] * tw, 0)
        s['st_ag_by_opp'] = s['st_ag_by_opp'] + jnp.sum(by_op[:N])
        s['st_opp_by_ag'] = s['st_opp_by_ag'] + jnp.sum(by_ag[N:])
        s['st_oob'] = s['st_oob'] + jnp.sum(oob[:N])
        out = dict(oob=oob[:N], kills_opp=kills_opp, kills_fr=kills_fr, by_op=by_op[:N], by_ag=by_ag[:N],
                   event=jnp.any(oob) | jnp.any(killer >= 0))
        return s, out

    def _low_rewards(self, s, killer, kind, ostat):
        N = self.N
        s, o = self._combat(s, killer, kind)
        if self.mode == 'fight':
            rk = 1.0 + 0.5 * s['mis_rem'][:N] / jnp.maximum(s['mis_max'][:N], 1.0)
            ck = (0.5 + 0.5 * s['can_rem'][:N] / s['can_max'][:N]) + (0.5 + 0.5 * ostat)
            per = jnp.where(kind[None, :] == 1, rk[:, None], ck[:, None])
            r = jnp.sum(o['kills_opp'] * per, 1)
        else:
            r = jnp.zeros((N,))
        r = r - 2.0 * jnp.sum(o['kills_fr'], 1) - 2.0 * o['by_op'] - 5.0 * o['oob']
        if FRIENDLY_PUNISH:
            r = r - 2.0 * o['by_ag']
        if self.mode == 'escape' and ESC_DIST_REW:
            dd = self._planar(s)[0]
            lst, _, n = self._sorted_enemies(s, dd)
            for k in range(self.K):
                o_k = jnp.maximum(lst[:N, k], 0)
                dk = dd[jnp.arange(N), o_k]
                valid = (n[:N] > k) & s['alive'][:N]
                spd = s['spd'][:N]
                near = (dk < 0.06) * (-0.02 / (k + 1)) * (1.0 + (spd < 200))
                far = (dk > 0.13) * (0.02 / (k + 1)) * (1.0 + (spd > 500))
                r = r + jnp.where(valid, near + far, 0.0)
        incl = s['alive'][:N] | o['oob'] | o['by_op'] | (FRIENDLY_PUNISH & o['by_ag'])
        return s, jnp.where(incl, r, 0.0), o['event']

    def _hl_rewards(self, s, killer, kind):
        N = self.N
        s, o = self._combat(s, killer, kind)
        r = 1.0 * jnp.sum(o['kills_opp'], 1) - 1.0 * o['by_op'] + self.oob_pen * o['oob']
        incl = s['alive'][:N] | o['oob'] | o['by_op']
        return s, jnp.where(incl, r, 0.0), o['event']

    def _done(self, s):
        na = jnp.sum(s['alive'][:self.N])
        no = jnp.sum(s['alive'][self.N:])
        done = (na == 0) | (no == 0) | (s['steps'] >= self.horizon)
        info = dict(win=(no == 0) & (s['steps'] < self.horizon), loss=(na == 0) & (s['steps'] < self.horizon),
                    draw=(s['steps'] >= self.horizon) & (na > 0) & (no > 0))
        return done, info

    # ------------------------------------------------------------------ step: low level
    def step(self, s, act, key, opp_params=()):
        if self.hl:
            return self.hl_step(s, act, key, opp_params)
        N, T = self.N, self.T
        ks = jax.random.split(key, 3)
        s = dict(s)
        s['steps'] = s['steps'] + 1
        alive_act = s['alive']
        a = jnp.zeros((T, 4), jnp.int32).at[:N].set(act.astype(jnp.int32))
        dd, foc, _ = self._planar(s)
        tg = jnp.maximum(s['tgt'][:N], 0)
        ostat = jnp.where(s['tgt'][:N] >= 0, jnp.clip(foc[tg, jnp.arange(N)] / 180.0, 0, 1), 0.0)  # opp_stats
        s, pen = self._base_action(s, alive_act & self.is_ag, a, s['tgt'], ks[0])
        pen = pen[:N] if self.mode == 'escape' else jnp.zeros((N,))
        if self.level >= 4:
            s = self._fsp_opps(s, ks[1], opp_params)
        else:
            s = self._scripted_opps(s, ks[1])
        s, killer, kind = self._tick(s, ks[2])
        s, r, _ = self._low_rewards(s, killer, kind, ostat)
        rew = jnp.where(alive_act[:N], pen + r, 0.0)
        obs, s = self._low_obs(s)
        done, info = self._done(s)
        trace = self._trace(s) if self.record else None
        return s, obs, rew, done, info, trace

    def _trace(self, s):
        return {k: s[k] for k in ('lat', 'lon', 'hdg', 'alive', 'r_lat', 'r_lon', 'r_alive', 'burst')}

    # ------------------------------------------------------------------ step: commander level
    def hl_step(self, s, cmd, key, llp):
        """One commander step. cmd: [N] commander actions (0 escape, k attack k-th nearest opponent).
        llp: dict(fight_ag=params, fight_opp=params, escape=params) of frozen low-level policies."""
        N, T = self.N, self.T
        ks = jax.random.split(key, 6)
        dd, foc, _ = self._planar(s)
        ar = jnp.arange(N)
        lst, n = s['lst'], s['lst_n']
        a = cmd.astype(jnp.int32)
        inval = (a > 0) & (a > n[:N])
        ca_ag = jnp.where(inval, 1, a)
        tg = jnp.maximum(lst[ar, jnp.maximum(ca_ag - 1, 0)], 0)
        cl = jnp.maximum(lst[ar, 0], 0)
        r_f = jnp.where((dd[ar, tg] < 0.1) & (foc[ar, tg] < 15) & (foc[tg, ar] > 40), 0.1, 0.0)
        r_e = jnp.where((dd[cl, ar] < 0.1) & (foc[cl, ar] < 15) & (foc[ar, cl] > 40), 0.1, 0.0)
        r0 = jnp.where(a > 0, r_f, r_e) if HIER_ACTION_ASSESS else jnp.zeros((N,))
        r0 = jnp.where(inval, -0.1, r0)
        rew = jnp.where(s['alive'][:N], r0, 0.0)
        # opponents: fight with prob ratio; attack a random non-nearest agent with prob 0.75
        no = n[N:]
        fight = jax.random.uniform(ks[0], (self.M,)) < HIER_OPP_FIGHT_RATIO / 100.0
        other = (no > 1) & (jax.random.uniform(ks[1], (self.M,)) < 0.75)
        pick = jax.random.randint(ks[2], (self.M,), 2, jnp.maximum(no, 2) + 1)
        ca_op = jnp.where(fight, jnp.where(other, pick, 1), 0)
        ca = jnp.concatenate([ca_ag, ca_op])
        s = dict(s)
        s['ca'] = jnp.where(s['alive'], ca, -1)

        def sub(c, idx):
            s, rew, active, key = c
            key, k1 = jax.random.split(key)
            s2, r_t, event = self._hl_substep(s, k1, llp)
            situation = (idx > 10) & self._surrounding(s2)
            s2['steps'] = s2['steps'] + 1
            s = tree_where(active, s2, s)
            rew = jnp.where(active, rew + r_t, rew)
            tr = (self._trace(s), active) if self.record else None
            return (s, rew, active & ~event & ~situation, key), tr

        (s, rew, _, _), trace = jax.lax.scan(sub, (s, rew, jnp.array(True), ks[3]), jnp.arange(16))
        obs, s = self._cmd_obs(s)
        done, info = self._done(s)
        return s, obs, rew, done, info, trace

    def _surrounding(self, s):
        dd, foc, _ = self._planar(s)
        N = self.N
        pair = s['alive'][:N, None] & s['alive'][None, N:]
        ev = (dd[:N, N:] < 0.1) & ((foc[:N, N:] < 15) | (foc[N:, :N].T < 15))
        return jnp.any(pair & ev)

    def _hl_substep(self, s, key, llp):
        N, T = self.N, self.T
        ks = jax.random.split(key, 3)
        P = self._planar(s)
        ar = jnp.arange(T)
        ca, lst, lst_dn, n = s['ca'], s['lst'], s['lst_dn'], s['lst_n']
        ci = jnp.maximum(ca - 1, 0)
        tgt_f = jnp.where(ca >= 1, lst[ar, ci], -1)
        tdn_f = lst_dn[ar, ci]
        far = lst[ar, jnp.maximum(n - 1, 0)]                   # escape: missile at the farthest (list[-1]) quirk
        mtgt = jnp.where(ca >= 1, tgt_f, jnp.where(ca == 0, far, -1))
        fr = self._sorted_friends(s, P[0])[:, 0]              # nearest living teammate, recomputed each tick
        use_esc = ca == 0

        def acts(s, rows, fparams):
            flags = self._shot(s)
            of = self._fight_rows(s, P, tgt_f, tdn_f, fr, flags)[rows]
            oe = self._esc_rows(s, P, lst, lst_dn, n, fr, flags)[rows]
            return frozen_ll_actions(fparams, llp['escape'], of, oe, s['ac1'][rows], use_esc[rows])

        a_ag = acts(s, slice(0, N), llp['fight_ag'])
        a = jnp.zeros((T, 4), jnp.int32).at[:N].set(a_ag)
        s, _ = self._base_action(s, s['alive'] & self.is_ag, a, mtgt, ks[0])
        a_op = acts(s, slice(N, T), llp['fight_opp'])          # opponents see agents' current shot flags
        a = jnp.zeros((T, 4), jnp.int32).at[N:].set(a_op)
        s, _ = self._base_action(s, s['alive'] & ~self.is_ag, a, mtgt, ks[1])
        s, killer, kind = self._tick(s, ks[2])
        return self._hl_rewards(s, killer, kind)


def make_batched(env):
    """jit+vmap reset/step for a batch of envs; step auto-resets finished envs."""
    reset_b = jax.vmap(env.reset)

    def step_b(s, act, keys, params):
        return jax.vmap(env.step, in_axes=(0, 0, 0, None))(s, act, keys, params)
    return reset_b, step_b

Quick sanity check of the simulator: random agent actions vs the scripted L3 opponents (no networks needed).

In [ ]:
def _env_sanity(n_envs=64, steps=300):
    env = Env('low', level=3, mode='fight')
    reset_b, step_b = make_batched(env)
    key = jax.random.PRNGKey(0)
    s, obs = jax.jit(reset_b)(jax.random.split(key, n_envs))

    @jax.jit
    def run(s, key):
        def body(c, k):
            s, ret, dn = c
            k1, k2 = jax.random.split(k)
            a = jax.random.randint(k1, (n_envs, 2, 4), 0, jnp.array([13, 9, 2, 2]))
            s2, obs, r, d, info, _ = step_b(s, a, jax.random.split(k2, n_envs), ())
            live = ~dn
            return (tree_where(live, s2, s), ret + jnp.where(live[:, None], r, 0), dn | d), (obs, info['win'] & live & d)
        (s, ret, dn), (obs, wins) = jax.lax.scan(body, (s, jnp.zeros((n_envs, 2)), jnp.zeros(n_envs, bool)),
                                                 jax.random.split(key, steps))
        return s, ret, dn, obs, wins
    t0 = time.time()
    s, ret, dn, obs, wins = jax.block_until_ready(run(s, key))
    print(f'L3 random-vs-scripted: {n_envs} envs x {steps} steps in {time.time() - t0:.1f}s (incl. compile); '
          f'done={int(dn.sum())}, agent wins={int(wins.sum())}, mean team return={float(ret.sum(1).mean()):.2f}, '
          f'obs range=[{float(obs.min()):.2f}, {float(obs.max()):.2f}], rockets fired={int(s["seq_ctr"].sum())}')

_env_sanity(8 if SMOKE else 256)